# Challenge 9: From High-Dimensional Survey Data to Pulsar Classification

**Author:** Dr Rob Lyon

**Version:** 1.0

## Code & License
Copyright &copy; 2026 Robert Lyon. All Rights Reserved.

Permission is granted to use, copy, and modify this material for
personal educational purposes only. Redistribution, resale, or use
in commercial or institutional settings requires prior written
permission from the author. This material is provided for educational
purposes as part of a structured course. The author accepts no
liability for incorrect results or decisions arising from use of this
material. All datasets used in this challenge are synthetic and do
not represent proprietary or confidential experimental data.

## Table of Contents

1. [Introduction](#1-introduction)
2. [Useful Links](#2-useful-links)
3. [Libraries and Environment Setup](#3-libraries-and-environment-setup)
   - [3.1 Working Environment](#31-working-environment)
   - [3.2 Libraries Used](#32-libraries-used)
   - [3.3 Data](#33-data)
   - [3.4 Imports](#34-imports)
4. [The Data](#4-the-data)
   - [4.1 Loading the Data](#41-loading-the-data)
   - [4.2 Understanding the Features](#42-understanding-the-features)
   - [4.3 Exploring the Data](#43-exploring-the-data)
5. [Step 1: Diagnosing the Dimensionality](#5-step-1-diagnosing-the-dimensionality)
   - [5.1 Feature Correlations](#51-feature-correlations)
   - [5.2 Baseline MLP on Raw Features](#52-baseline-mlp-on-raw-features)
6. [Step 2: Applying PCA](#6-step-2-applying-pca)
   - [6.1 Preprocessing for PCA](#61-preprocessing-for-pca)
   - [6.2 The Scree Plot and Explained Variance](#62-the-scree-plot-and-explained-variance)
   - [6.3 Visualising the Data in PCA Space](#63-visualising-the-data-in-pca-space)
   - [6.4 Understanding the Principal Components](#64-understanding-the-principal-components)
7. [Step 3: Classifying in the Reduced Space](#7-step-3-classifying-in-the-reduced-space)
   - [7.1 MLP on PCA-Reduced Features](#71-mlp-on-pca-reduced-features)
   - [7.2 Comparing Across Component Counts](#72-comparing-across-component-counts)
   - [7.3 Reflection Questions](#73-reflection-questions)
8. [Solution](#8-solution)
9. [References](#9-references)

---
## 1. Introduction

The Parkes High Time Resolution Universe (HTRU) survey processed
hundreds of thousands of radio frequency candidates in search of new
pulsars. Each candidate is the output of a folding pipeline: the survey
telescope records the sky, a search algorithm finds a periodic signal at
a candidate period and dispersion measure, and the data for that period
and DM are folded and averaged to produce a candidate signal. The vast
majority of candidates are not pulsars: they are radio frequency
interference from human technology, or they are noise fluctuations that
the periodicity search algorithm mistook for a periodic signal.

The HTRU-2 dataset (Lyon et al. 2016) - yes, that's my data - contains 17,898 such candidates
described by 8 statistical features computed from the folded pulse profile
and the dispersion measure vs signal-to-noise ratio curve. Approximately
9% of candidates are genuine pulsars. A machine learning classifier trained
on these 8 features can separate pulsars from non-pulsars with high
accuracy, but modern pulsar survey pipelines compute far more than 8
features: sub-band profile statistics, higher-order spectral scores,
frequency coherence metrics, and polarisation diagnostics all add
dimensions to the feature space.

The **RadioSurvey** dataset in this challenge extends the HTRU-2 feature
set to 20 features by adding 12 correlated additional scores that
simulate what a modern pipeline produces. The 8 original features carry
the genuine pulsar signal; the 12 additional features are correlated
noisy versions of the originals that add dimensions without adding
independent class information.

This challenge takes you through a complete pipeline from problem to
diagnosis to dimensionality reduction to classification:

1. **Diagnose**: explore the 20-feature space, examine correlations,
   establish a baseline Multi-layer Perceptron (MLP) on raw features
2. **Reduce**: apply Principal Components Analysis (PCA), choose the number of components using the
   scree plot, visualise the data in 2D PCA space
3. **Classify**: train an MLP in the reduced PCA space and compare it
   to the raw-feature baseline

The key insight this challenge is designed to demonstrate: the 20 raw
features contain mostly redundant information. PCA finds that 2-4
components capture the vast majority of the class-relevant structure.
An MLP trained on 2 PCA components achieves nearly the same accuracy
as an MLP trained on all 20 raw features, but the 2-component space
can be visualised directly, the model is more interpretable, and the
pipeline is more robust to new features being added by future survey
software.

### Learning Objectives

- Diagnose a high-dimensional feature space using correlation analysis
  and identify redundant features
- Apply `StandardScaler` correctly before PCA and understand why
  scaling is mandatory for PCA but not for tree-based methods
- Use a scree plot and cumulative explained variance to choose the
  number of PCA components
- Visualise a high-dimensional dataset in 2D using PCA and interpret
  the principal components in terms of the original features
- Train and evaluate an `MLPClassifier` on PCA-reduced features and
  compare to the raw-feature baseline
- Understand that PCA maximises explained variance, not explained class
  signal, and reason about the implications for choosing n\_components

---
## 2. Useful Links

| Resource | URL |
|---|---|
| `PCA` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html |
| PCA user guide (scikit-learn) | https://scikit-learn.org/stable/modules/decomposition.html#pca |
| `MLPClassifier` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html |
| `StandardScaler` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html |
| `classification_report` (scikit-learn) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html |
| HTRU-2 dataset (UCI repository) | https://archive.ics.uci.edu/dataset/372/htru2 |
| Lyon et al. (2016) HTRU-2 paper | https://doi.org/10.1093/mnras/stw656 |
| Eatough et al. (2009) pulsar feature definitions | https://doi.org/10.1111/j.1365-2966.2009.15100.x |

---
## 3. Libraries and Environment Setup

### 3.1 Working Environment

This notebook requires Python 3.9 or later and scikit-learn 1.0 or later.
All operations complete in seconds on a modern laptop.

### 3.2 Libraries Used

| Library | Purpose |
|---|---|
| `numpy` | Numerical operations |
| `pandas` | Loading and inspecting the dataset |
| `matplotlib` | Plotting (scree plots, scatter plots, component loadings) |
| `seaborn` | Correlation heatmaps and distribution plots |
| `sklearn.preprocessing` | `StandardScaler` for feature scaling before PCA |
| `sklearn.decomposition` | `PCA` for dimensionality reduction |
| `sklearn.neural_network` | `MLPClassifier` |
| `sklearn.model_selection` | `train_test_split` with stratification |
| `sklearn.metrics` | `classification_report`, `f1_score`, `accuracy_score` |

### 3.3 Data

| Property | Value |
|---|---|
| Filename | `radio.csv` |
| Location | `data/radio.csv` (relative to this notebook) |
| Total rows | 10,000 |
| Features | 20 (8 signal features + 12 correlated additional scores) |
| Target column | `candidate_type` |
| Class 0 | non\_pulsar: approx. 8,950 samples (approx. 89.5%) |
| Class 1 | pulsar: approx. 1,050 samples (approx. 10.5%) |

The dataset is imbalanced: approximately 1 in 10 candidates is a genuine
pulsar, reflecting the real HTRU survey composition. This means overall
accuracy is a misleading metric: a classifier that always predicted
non-pulsar would achieve ~89.5% accuracy. Pulsar recall (sensitivity)
and macro-averaged F1 score are the appropriate metrics here.

### 3.4 Imports

In [ ]:
# Listing 1.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

CLASS_NAMES   = ['non_pulsar', 'pulsar']
SIGNAL_FEATS  = ['prof_mean', 'prof_std', 'prof_skewness', 'prof_kurtosis',
                 'dm_snr_mean', 'dm_snr_std', 'dm_snr_skewness', 'dm_snr_kurtosis']
EXTRA_FEATS   = ([f'subband_{i:02d}' for i in range(1, 9)] +
                 [f'pipeline_{i:02d}' for i in range(1, 5)])
FEATURE_COLS  = SIGNAL_FEATS + EXTRA_FEATS

print('Libraries loaded successfully.')
print(f'Total features: {len(FEATURE_COLS)}  '
      f'({len(SIGNAL_FEATS)} signal + {len(EXTRA_FEATS)} extra)')

---
## 4. The Data

### 4.1 Loading the Data

In [ ]:
# Listing 2.
df = pd.read_csv('data/radio.csv')

print(f'Dataset shape: {df.shape}')
print(f'Columns: {df.columns.tolist()}')
print()
df.head(8)

### 4.2 Understanding the Features

The 8 signal features are the direct equivalents of the HTRU-2 features
defined by Eatough et al. (2009). They are statistical summaries of two
diagnostic curves computed for every pulsar candidate.

**Profile statistics** (statistics of the folded pulse profile):

| Feature | Description |
|---|---|
| `prof_mean` | Mean of the integrated (folded) pulse profile. Pulsars typically have elevated means from the concentrated pulse energy. |
| `prof_std` | Standard deviation of the integrated profile. Narrow pulsar profiles have moderate std; broad RFI profiles have higher std. |
| `prof_skewness` | Excess kurtosis of the profile (note: named skewness for historical reasons). Real pulsars have highly peaked profiles. |
| `prof_kurtosis` | Skewness of the profile. Real pulsars have asymmetric profiles (sharp rise, slower fall). |

**DM-SNR curve statistics** (statistics of the signal-to-noise ratio as a function of trial dispersion measure):

| Feature | Description |
|---|---|
| `dm_snr_mean` | Mean of the DM-SNR curve. |
| `dm_snr_std` | Standard deviation of the DM-SNR curve. Real pulsars have concentrated peaks (low std); RFI has flat or multi-peaked curves. |
| `dm_snr_skewness` | Excess kurtosis of the DM-SNR curve. Sharp DM peaks have high kurtosis. |
| `dm_snr_kurtosis` | Skewness of the DM-SNR curve. Real pulsars often show asymmetric DM peaks. |

**Additional features** (12 features from the extended pipeline):

| Feature group | Description |
|---|---|
| `subband_01` to `subband_08` | Profile and DM-SNR statistics computed in individual frequency sub-bands of the receiver. Correlated with the signal features but add noise. |
| `pipeline_01` to `pipeline_04` | Higher-order diagnostic scores from the survey pipeline. Weak linear combinations of all signal features plus noise. |

### 4.3 Exploring the Data

In [ ]:
# Listing 3.
print('Class distribution:')
counts = df['candidate_type'].value_counts().sort_index()
for k, v in counts.items():
    print(f'  Class {k} ({CLASS_NAMES[k]:12s}): {v:5d}  ({100*v/len(df):.1f}%)')

print()
print('Feature summary statistics:')
print(df[SIGNAL_FEATS].describe().round(3).to_string())

**Questions to consider:**

- With 89.5% non-pulsars, what accuracy would a trivial classifier achieve
  by predicting "non-pulsar" for every candidate? Why does this make
  accuracy a bad headline metric for this problem?
- Which evaluation metric should you use to ensure the pulsar class
  (the minority class you actually care about) is well-characterised?
- Look at the feature means and standard deviations. Do the distributions
  appear similar between signal features and extra features?

In [ ]:
# Listing 4.
# Compare distributions for 4 signal features
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
axes = axes.flatten()
colours = {0: 'steelblue', 1: 'firebrick'}

for ax, feat in zip(axes, SIGNAL_FEATS):
    for cls in [0, 1]:
        subset = df.loc[df['candidate_type'] == cls, feat]
        subset.plot(kind='kde', ax=ax, label=CLASS_NAMES[cls],
                    color=colours[cls], linewidth=2)
    ax.set_title(feat, fontsize=10)
    ax.set_xlabel('')
    ax.legend(fontsize=9)

plt.suptitle('Signal Feature Distributions by Candidate Type (RadioSurvey)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Questions to consider:**

- Which features show the clearest separation between pulsars and
  non-pulsars? Do the two classes overlap substantially in any features?
- The two DM-SNR kurtosis features (`dm_snr_skewness`, `dm_snr_kurtosis`)
  appear to separate pulsars (high values) from non-pulsars. Does this
  make physical sense? (Hint: a sharp DM peak at the true DM should produce
  high kurtosis in the DM-SNR curve.)
- Given that some features clearly separate the two classes, do you expect
  a simple MLP to perform well even on the raw 20 features?

---
## 5. Step 1: Diagnosing the Dimensionality

### 5.1 Feature Correlations

In [ ]:
# Listing 5.
# Correlation matrix
corr = df[FEATURE_COLS].corr()

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(corr, annot=False, cmap='coolwarm', vmin=-1, vmax=1,
            linewidths=0.3, ax=ax, square=True)
ax.set_title('Feature Correlation Matrix (RadioSurvey, 20 features)', fontsize=12)

# Draw rectangle around signal features
ax.add_patch(plt.Rectangle((0, 0), 8, 8,
             fill=False, edgecolor='black', linewidth=2))
ax.text(4, -0.5, 'Signal features (8)', ha='center', va='top', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

**Questions to consider:**

- The correlation matrix has a visible block structure. The signal features
  (top-left 8x8 block) show moderate correlations with each other. The
  extra features (subband and pipeline scores) are correlated with the
  signal features because they were derived from them. What does this
  tell you about the effective dimensionality of the dataset?
- If many features carry similar information (high correlation), how
  many truly independent dimensions does the dataset have?
- What would you expect a PCA analysis to show about the number of
  components needed to capture most of the variance?

### 5.2 Baseline MLP on Raw Features

Before applying any dimensionality reduction, establish a baseline by
training an MLP on the full 20-feature scaled dataset. This gives you
a benchmark to compare against after PCA.

Because the dataset is imbalanced (89.5% non-pulsar), note that
`MLPClassifier` does not support `class_weight`. Instead, evaluate using
macro-averaged F1 score and pay close attention to pulsar recall. A model
that predicts "non-pulsar" for everything would score F1=0 for the pulsar
class.

In [ ]:
# Listing 6.
# TODO: Baseline MLP on raw scaled features.
#
# 1. Separate features (X) from the target ('candidate_type') (y).
#    Use an 80/20 stratified split, random_state=42.
#    Print class counts in y_train and y_test.
#
# 2. Apply StandardScaler to X.
#    Fit on X_train, transform both X_train and X_test.
#    Store as X_train_sc and X_test_sc.
#
# 3. Instantiate MLPClassifier with:
#      hidden_layer_sizes=(64, 32)
#      max_iter=500
#      random_state=42
#      early_stopping=True
#      validation_fraction=0.1
#    Fit on X_train_sc.
#
# 4. Compute and print:
#    - Overall accuracy on X_test_sc
#    - Macro-averaged F1 score
#    - Full classification_report with target_names=CLASS_NAMES
#    - The number of iterations until convergence (mlp.n_iter_)
#
# 5. Plot the confusion matrix. Note how many genuine pulsars are missed
#    (false negatives in the pulsar row).

# YOUR CODE HERE

---
## 6. Step 2: Applying PCA

### 6.1 Preprocessing for PCA

PCA finds the directions of maximum variance in the data. If features are
on different scales, the features with large variance will dominate the
principal components regardless of their discriminating power. This is why
PCA requires standardised input: `StandardScaler` must be applied before
fitting PCA. You already fit the scaler in Section 5.2; use the same
scaler here to transform both training and test sets.

This is also why PCA differs from tree-based methods: decision trees and
random forests split on individual feature thresholds and are invariant
to monotonic scaling. PCA is not: it treats all features symmetrically by
their variance, so scale matters directly and cannot be skipped.

In [ ]:
# Listing 7.
# TODO: Fit PCA on the scaled training data.
#
# 1. Instantiate PCA(random_state=42) with no n_components limit.
#    Fit on X_train_sc (already computed in Section 5.2).
#    This fits a full PCA with all 20 components.
#
# 2. Print pca.explained_variance_ratio_ (the fraction of variance
#    explained by each component).
#    Print np.cumsum(pca.explained_variance_ratio_) for components 1-10.
#
# 3. How many components are needed to explain 80% of the variance?
#    How many for 90%? Store the 80% threshold as n_components_80.

# YOUR CODE HERE

### 6.2 The Scree Plot and Explained Variance

In [ ]:
# Listing 8.
# TODO: Plot the scree plot and cumulative explained variance.
#
# Create a figure with two panels side by side:
#
# Left panel: Individual explained variance ratio by component (bar chart).
#   - x-axis: Component number (1 to 20)
#   - y-axis: Explained variance ratio (fraction)
#   - Title: 'Scree Plot'
#   - Add a horizontal dashed line at the mean variance per component
#     (1/20 = 0.05). Components above this line carry above-average variance.
#
# Right panel: Cumulative explained variance (line plot).
#   - x-axis: Number of components
#   - y-axis: Cumulative explained variance ratio
#   - Add horizontal dashed lines at 0.80 and 0.90
#   - Mark the point where the curve crosses 80% with a vertical line
#   - Title: 'Cumulative Explained Variance'
#
# After plotting, answer in a markdown cell:
#   - Is there a clear "elbow" in the scree plot?
#   - How many components does the 80% threshold suggest?
#   - What does it mean that there is no sharp elbow in this dataset?

# YOUR CODE HERE

### 6.3 Visualising the Data in PCA Space

One of the most powerful uses of PCA is visualisation: projecting a
high-dimensional dataset onto its first two or three principal components
allows you to see whether the classes form distinct clusters. For pulsar
classification, a 2D PCA scatter plot is a direct visual diagnostic:
if genuine pulsars form a compact cluster separated from non-pulsars in
the PC1-PC2 plane, it confirms that the feature set carries strong signal
and that 2 components may be sufficient for a competent classifier.

In [ ]:
# Listing 9.
# TODO: Visualise the dataset in 2D PCA space.
#
# 1. Transform X_train_sc and X_test_sc using the fitted PCA.
#    (pca.transform() applies the full 20-component projection;
#     then select only the first 2 columns for visualisation.)
#
# 2. Create a scatter plot of PC1 vs PC2 coloured by true class label.
#    Use the test set (X_test_sc) for clarity (smaller, faster to plot).
#    Use a different colour and marker for each class.
#    Alpha=0.5 and s=12 for visibility.
#    Label axes 'PC1 (X.X% variance)' using the actual variance fraction.
#    Title: '2D PCA Projection: RadioSurvey'
#
# 3. Do pulsars form a distinct cluster in the PC1-PC2 space?
#    Are there any pulsars embedded deep inside the non-pulsar region?
#    What does the scatter plot tell you about the difficulty of the
#    classification problem?
#
# 4. Optionally: repeat for PC2 vs PC3 to see if the third component
#    adds additional class separation.

# YOUR CODE HERE

### 6.4 Understanding the Principal Components

The principal components are linear combinations of the original features.
The loadings (or eigenvectors) stored in `pca.components_` tell you how
much each original feature contributes to each principal component.
`pca.components_[0]` is the first principal component: the direction of
maximum variance. Inspecting the loadings tells you which original features
drive the first few components, and whether those features are the ones
you expect to carry class information.

In [ ]:
# Listing 10.
# TODO: Inspect the component loadings.
#
# 1. Create a DataFrame from pca.components_[:6, :] (first 6 components).
#    Rows = PC1 through PC6, columns = FEATURE_COLS.
#    Print it with 3 decimal places.
#
# 2. Create a heatmap of the first 6 component loadings.
#    Use a diverging colourmap (coolwarm). Rows = PCs, columns = features.
#    Title: 'PCA Component Loadings (first 6 PCs)'
#
# 3. Look at PC1: which original features have the largest absolute loadings?
#    Are they the signal features or the extra features?
#    Does PC1 primarily represent the signal or the noise in the data?
#
# 4. Look at PC2: does it load on different features than PC1?
#    Together, do PC1 and PC2 cover the most discriminating features
#    (dm_snr_kurtosis, prof_kurtosis) that separated pulsars in the KDE
#    plots from Section 4.3?

# YOUR CODE HERE

---
## 7. Step 3: Classifying in the Reduced Space

### 7.1 MLP on PCA-Reduced Features

Now train the MLP on the PCA-reduced data and compare to the raw-feature
baseline. The key question is not whether PCA improves accuracy (it may
not, because the MLP can implicitly learn to ignore redundant features)
but rather:

- Does the MLP trained on 2-4 PCA components achieve comparable accuracy
  to the full 20-feature MLP?
- If so, what have we gained by using PCA?

If 2 components give comparable accuracy to 20 features, the answer is
significant: we have shown that the effective dimensionality of the
problem is 2, that we can visualise the entire classification problem
in 2D, that the model is far simpler and more interpretable, and that
we have a principled understanding of what the classifier is doing in
physical terms (it is primarily using DM-SNR kurtosis and profile
kurtosis, as revealed by the component loadings).

In [ ]:
# Listing 11.
# TODO: MLP on PCA-reduced features.
#
# 1. Choose a number of components. Start with the 80% variance threshold
#    you computed in Section 6.1. Also try n_components=2 (for comparison
#    with the visualisation) and n_components=4.
#
# 2. For your chosen n_components:
#    - Create a new PCA(n_components=n_comp, random_state=42)
#    - Fit on X_train_sc, transform both X_train_sc and X_test_sc
#    - Train the same MLPClassifier architecture as Section 5.2
#    - Print accuracy, macro-F1, classification report, and n_iter_
#
# 3. Compare to the raw-feature baseline:
#    - How much did accuracy change?
#    - How much did pulsar recall change?
#    - How many iterations did the MLP need to converge?

# YOUR CODE HERE

### 7.2 Comparing Across Component Counts

To understand the tradeoff between dimensionality and classification
performance, sweep through different values of n\_components and record
the accuracy and pulsar F1 at each value. This curve tells you the
minimum number of PCA components needed to retain most of the
classification performance.

In [ ]:
# Listing 12.
# TODO: Accuracy vs n_components sweep.
#
# 1. Loop over n_components from 1 to 20.
#    For each value:
#    - Fit PCA with n_components on X_train_sc
#    - Transform train and test
#    - Train the same MLP
#    - Record accuracy, pulsar F1, and n_iter_
#
# 2. Plot three metrics vs n_components on the same figure (or subplots):
#    - Overall accuracy
#    - Pulsar F1 score (f1_score(y_test, y_pred, pos_label=1))
#    - Number of iterations to convergence (n_iter_)
#
# 3. Add a vertical dashed line at the 80% cumulative variance threshold.
#    Add another at n_components=2.
#
# 4. Where does performance plateau? Is there a point beyond which adding
#    more components does not improve (or even hurts) accuracy?
#    What does this tell you about the effective dimensionality of the
#    classification problem?

# YOUR CODE HERE

### 7.3 Reflection Questions

1. The scree plot for this dataset does not show a sharp elbow. The
   variance decreases gradually. What does this tell you about the
   correlation structure of the 20 features? Compare this to a dataset
   where you would expect a sharp elbow.

2. The MLP on 2 PCA components achieves nearly the same accuracy as
   the MLP on 20 raw features. This might surprise you: how can 2 numbers
   carry as much information as 20? Explain this in terms of the
   correlation structure of the feature set.

3. PCA maximises explained variance, not explained class signal. The
   first principal component explains the most variance in the data, but
   it may not be the direction that best separates pulsars from non-pulsars.
   Could there be a direction that better separates the two classes while
   explaining less total variance? What algorithm would find such a
   direction? (Hint: think about Linear Discriminant Analysis.)

4. In a real pulsar survey, new features might be added by the survey
   software as it is upgraded (new sub-band statistics, polarisation
   diagnostics, etc.). If you had trained an MLP directly on the raw
   features, adding a new feature would require retraining from scratch.
   How does the PCA pipeline handle this situation differently?

5. The pulsar class has only 10.5% of samples. The MLP is not told about
   this imbalance explicitly. Look at the pulsar recall in your best model:
   how many genuine pulsars are being missed? In a real survey context,
   what is the consequence of a missed pulsar? How might you adjust the
   classification threshold or the training procedure to improve recall?

---
## 8. Solution

**Read this section only after attempting the challenge yourself.**
Every step is explained with the physical reasoning behind it.

### Step 1: Baseline MLP on Raw Features

In [ ]:
# Listing 13.
X = df[FEATURE_COLS].values
y = df['candidate_type'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print('Class counts in training set:')
for cls in [0, 1]:
    n = int(np.sum(y_train == cls))
    print(f'  {CLASS_NAMES[cls]:12s}: {n:5d}  ({100*n/len(y_train):.1f}%)')

# Baseline MLP
mlp_raw = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500,
                         random_state=42, early_stopping=True,
                         validation_fraction=0.1)
mlp_raw.fit(X_train_sc, y_train)
y_pred_raw = mlp_raw.predict(X_test_sc)

acc_raw = accuracy_score(y_test, y_pred_raw)
f1_raw  = f1_score(y_test, y_pred_raw, average='macro')

print(f'\nBaseline MLP (20 raw features):')
print(f'  Accuracy:      {acc_raw:.4f}')
print(f'  Macro-F1:      {f1_raw:.4f}')
print(f'  Iterations:    {mlp_raw.n_iter_}')
print()
print(classification_report(y_test, y_pred_raw, target_names=CLASS_NAMES))

The raw-feature MLP achieves high overall accuracy because the non-pulsar
class dominates. The pulsar recall is the critical metric: it tells us
what fraction of genuine pulsars the model correctly identifies. In the
context of a pulsar survey, a missed pulsar is a missed scientific
discovery: the candidate will be discarded and never followed up.

Note also the number of iterations to convergence. We will compare this
to the PCA-based MLP to see whether working in a lower-dimensional space
changes convergence behaviour.

In [ ]:
# Listing 14.
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred_raw,
    display_labels=CLASS_NAMES, cmap='Blues', ax=ax
)
ax.set_title('Baseline MLP (20 raw features)', fontsize=10)
plt.tight_layout()
plt.show()

### Step 2: PCA Analysis

In [ ]:
# Listing 15.
# Fit full PCA on scaled training data
pca_full = PCA(random_state=42)
pca_full.fit(X_train_sc)

evr    = pca_full.explained_variance_ratio_
cumvar = np.cumsum(evr)

print('Explained variance by component:')
for i, (v, cv) in enumerate(zip(evr[:12], cumvar[:12]), 1):
    print(f'  PC{i:2d}: {v:.4f}  (cumulative: {cv:.4f})')

n_80 = int(np.argmax(cumvar >= 0.80)) + 1
n_90 = int(np.argmax(cumvar >= 0.90)) + 1
print(f'\nComponents to reach 80% variance: {n_80}')
print(f'Components to reach 90% variance: {n_90}')

### Step 3: Scree Plot and Cumulative Variance

In [ ]:
# Listing 16.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Scree plot
ax1.bar(range(1, 21), evr, color='steelblue', alpha=0.8)
ax1.axhline(1/20, color='firebrick', linestyle='--', linewidth=1.5,
            label=f'Mean variance (1/20 = {1/20:.3f})')
ax1.set_xlabel('Principal Component', fontsize=11)
ax1.set_ylabel('Explained Variance Ratio', fontsize=11)
ax1.set_title('Scree Plot', fontsize=12)
ax1.legend(fontsize=9)

# Cumulative variance
ax2.plot(range(1, 21), cumvar, 'o-', color='steelblue', linewidth=2)
ax2.axhline(0.80, color='goldenrod', linestyle='--', linewidth=1.5, label='80% threshold')
ax2.axhline(0.90, color='firebrick', linestyle='--', linewidth=1.5, label='90% threshold')
ax2.axvline(n_80, color='goldenrod', linestyle=':', linewidth=1.2)
ax2.axvline(2,    color='seagreen',  linestyle=':', linewidth=1.2, label='n=2 (visualisation)')
ax2.set_xlabel('Number of Components', fontsize=11)
ax2.set_ylabel('Cumulative Explained Variance', fontsize=11)
ax2.set_title('Cumulative Explained Variance', fontsize=12)
ax2.legend(fontsize=9)

plt.suptitle('PCA Variance Analysis: RadioSurvey', fontsize=13)
plt.tight_layout()
plt.show()

**Reading the scree plot:**

There is no sharp elbow in this scree plot. Variance decreases gradually
across all 20 components. This is characteristic of a dataset where many
features are correlated mixtures of a smaller number of underlying signals:
each component picks up a bit more of the shared structure, and none
dramatically dominates the rest after the first component.

In contrast, a dataset with truly independent low-dimensional structure
would show a sharp drop after the first few components, with all subsequent
components near the mean (the horizontal dashed line). When the scree plot
is gradual, the 80% or 90% variance threshold is a pragmatic rule of thumb
rather than a principled elbow.

For classification purposes, the variance threshold may overestimate the
number of needed components. The classification boundary may be captured
in the first 2-4 components even if those components explain only 40-50%
of the total variance. We will see this directly in the accuracy sweep.

### Step 4: 2D Visualisation

In [ ]:
# Listing 17.
X_test_2d = pca_full.transform(X_test_sc)[:, :2]

fig, ax = plt.subplots(figsize=(9, 6))
colours = {0: 'steelblue', 1: 'firebrick'}
labels_text = {0: 'Non-pulsar', 1: 'Pulsar'}
for cls in [0, 1]:
    mask = y_test == cls
    ax.scatter(X_test_2d[mask, 0], X_test_2d[mask, 1],
               c=colours[cls], alpha=0.4, s=12,
               label=f'{labels_text[cls]} (n={mask.sum()})')

ax.set_xlabel(f'PC1 ({evr[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({evr[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('2D PCA Projection: RadioSurvey Pulsar Candidates', fontsize=12)
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

The 2D scatter plot is the most important visualisation in this challenge.
It shows that genuine pulsars (red points) form a compact cluster that is
largely separated from the non-pulsar candidates (blue points) in just
two dimensions, even though the original feature space has 20 dimensions.

This is the key finding: the effective dimensionality of the classification
problem is much lower than the number of features. The 20 measured
quantities collapse to a 2D representation that retains the class structure.
This is not a coincidence; it follows from the physical nature of the
features: they are all measuring aspects of the same two underlying
phenomena (the shape of the pulse and the shape of the DM curve), so the
true information content is low-dimensional from the start.

There is some overlap between the two classes at the boundary, which
explains why no classifier achieves perfect accuracy. That overlap
represents the genuinely ambiguous candidates: narrowband RFI that produces
narrow pulse profiles, or pulsars with unusual DM-SNR curves due to
interstellar scattering.

### Step 5: Component Loadings

In [ ]:
# Listing 18.
loadings_df = pd.DataFrame(
    pca_full.components_[:6, :],
    index=[f'PC{i+1}' for i in range(6)],
    columns=FEATURE_COLS
)
print('First 6 PCA component loadings:')
print(loadings_df.round(3).to_string())

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(loadings_df, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, ax=ax, annot_kws={'size': 7})
ax.set_title('PCA Component Loadings (first 6 components)', fontsize=12)
ax.set_xlabel('Feature')
ax.set_ylabel('Principal Component')
plt.tight_layout()
plt.show()

**Reading the loadings heatmap:**

Each row is a principal component; each column is an original feature.
The colour indicates the sign and magnitude of the contribution.

PC1 typically loads heavily on the extra features (sub-band and pipeline
scores) because those features have high shared variance from the common
calibration structure. This illustrates an important subtlety: the first
principal component explains the most variance, but it may not be the most
useful for classification. The extra features share variance through their
correlated construction, and PC1 may primarily capture that shared
structure rather than the pulsar signal.

The signal features (`dm_snr_kurtosis`, `prof_kurtosis`, etc.) that clearly
separated the two classes in the KDE plots of Section 4.3 should appear
with substantial loadings in PC1 or PC2. The extent to which they do
tells you whether PCA has found the class-discriminating directions.

### Step 6: MLP on PCA-Reduced Features

In [ ]:
# Listing 19.
# Accuracy sweep across n_components
n_comp_values = list(range(1, 21))
accs, f1s, n_iters = [], [], []

for nc in n_comp_values:
    pca_nc = PCA(n_components=nc, random_state=42)
    Xtr_p  = pca_nc.fit_transform(X_train_sc)
    Xte_p  = pca_nc.transform(X_test_sc)
    mlp_nc = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500,
                            random_state=42, early_stopping=True,
                            validation_fraction=0.1)
    mlp_nc.fit(Xtr_p, y_train)
    y_pred = mlp_nc.predict(Xte_p)
    accs.append(accuracy_score(y_test, y_pred))
    f1s.append(f1_score(y_test, y_pred, pos_label=1))
    n_iters.append(mlp_nc.n_iter_)
    if nc in [2, 4, n_80]:
        print(f'n_comp={nc:2d}  acc={accs[-1]:.4f}  pulsar-F1={f1s[-1]:.4f}  iters={n_iters[-1]}')

print(f'\nRaw 20 features:  acc={acc_raw:.4f}  pulsar-F1={f1_score(y_test,y_pred_raw,pos_label=1):.4f}  iters={mlp_raw.n_iter_}')

In [ ]:
# Listing 20.
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

ax1.plot(n_comp_values, accs, 'o-', color='steelblue', linewidth=2)
ax1.axhline(acc_raw, color='firebrick', linestyle='--', linewidth=1.5,
            label=f'Raw 20 features ({acc_raw:.4f})')
ax1.axvline(2,    color='seagreen',  linestyle=':', linewidth=1.2, label='n=2')
ax1.axvline(n_80, color='goldenrod', linestyle=':', linewidth=1.2, label=f'n={n_80} (80% var)')
ax1.set_ylabel('Accuracy', fontsize=11)
ax1.set_title('MLP Performance vs PCA Components: RadioSurvey', fontsize=12)
ax1.legend(fontsize=9)

ax2.plot(n_comp_values, f1s, 's-', color='seagreen', linewidth=2)
ax2.axhline(f1_score(y_test, y_pred_raw, pos_label=1),
            color='firebrick', linestyle='--', linewidth=1.5, label='Raw 20 features')
ax2.axvline(2,    color='seagreen',  linestyle=':', linewidth=1.2)
ax2.axvline(n_80, color='goldenrod', linestyle=':', linewidth=1.2)
ax2.set_ylabel('Pulsar F1 Score', fontsize=11)
ax2.legend(fontsize=9)

ax3.plot(n_comp_values, n_iters, '^-', color='darkorange', linewidth=2)
ax3.axhline(mlp_raw.n_iter_, color='firebrick', linestyle='--',
            linewidth=1.5, label='Raw 20 features')
ax3.set_xlabel('Number of PCA Components', fontsize=11)
ax3.set_ylabel('MLP Iterations to Convergence', fontsize=11)
ax3.legend(fontsize=9)

plt.tight_layout()
plt.show()

**Reading the performance curves:**

The accuracy and pulsar F1 curves typically show that:

- With only 2 PCA components, the MLP achieves accuracy and pulsar F1
  scores close to the raw-feature baseline. This confirms the 2D scatter
  plot finding: the class structure is largely captured in two dimensions.
- Performance may plateau after 3-5 components and then remain roughly
  constant or decline slightly as more components are added. The decline
  at high component counts can happen when the MLP starts fitting the noise
  structure captured by the later PCA components.
- The number of iterations to convergence is often lower in the reduced
  space: fewer parameters, simpler geometry, faster convergence.

The key practical conclusion: for this dataset, 2-4 PCA components are
sufficient to achieve near-optimal classification performance. Using all
20 features gives marginal improvement at best, and the 2-component model
has the decisive advantage of being directly visualisable, as you saw in
the 2D scatter plot.

This is not always the case. In problems where the class boundary is
genuinely high-dimensional (each feature adds independent discriminating
information), PCA reduction will cause accuracy to decrease monotonically
as components are removed. The fact that performance is stable across a
wide range of component counts here tells you that the 20 features are
measuring a small number of underlying phenomena, and that dimensionality
reduction is finding rather than discarding the informative structure.

---
## 9. References

1. Lyon, R.J., Stappers, B.W., Cooper, S., Brooke, J.M., and Knowles,
   J.D. (2016). Fifty years of pulsar candidate selection: from simple
   filters to a new principled real-time classification approach.
   *Monthly Notices of the Royal Astronomical Society*, 459(2), 1104-1123.
   https://doi.org/10.1093/mnras/stw656
   Defines the HTRU-2 dataset and the 8 statistical features used in this
   challenge, and reviews machine learning approaches to pulsar candidate
   classification.

2. Eatough, R.P., Molkenthin, N., Kramer, M., Noutsos, A., Johnston, S.,
   Stappers, B.W., and Lyne, A.G. (2009). Selection of radio pulsar
   candidates using artificial neural networks.
   *Monthly Notices of the Royal Astronomical Society*, 407(4), 2443-2450.
   https://doi.org/10.1111/j.1365-2966.2009.15100.x
   Original paper defining the profile and DM-SNR statistical features
   and their physical interpretation for pulsar candidate selection.

3. Pearson, K. (1901). On lines and planes of closest fit to systems of
   points in space. *Philosophical Magazine*, 2(11), 559-572.
   Original paper introducing Principal Component Analysis.

4. Bache, K. and Lichman, M. (2013). UCI Machine Learning Repository.
   University of California, Irvine.
   https://archive.ics.uci.edu/dataset/372/htru2
   The HTRU-2 dataset is publicly available from the UCI repository.

5. Jolliffe, I.T. (2002). *Principal Component Analysis* (2nd ed.).
   Springer Series in Statistics.
   Comprehensive reference for PCA theory, including the relationship
   between explained variance, component loadings, and dimensionality
   selection criteria.

6. Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python.
   *Journal of Machine Learning Research*, 12, 2825-2830.
   https://jmlr.org/papers/v12/pedregosa11a.html